# Getting started

In [ ]:
# Standard library imports
import configparser
import os

# Third-party imports
import numpy as np
import pandas as pd
from flask import Flask
import anthropic

# Pandas configuration
pd.set_option('display.max_colwidth', None)

## 1. Data

In [ ]:
data_sample = pd.read_csv("../data/twitter_data_clean_sample.csv")

In [ ]:
data_sample.head()

In [ ]:
data_sample.company.value_counts()

## 2. How to use the Anthropic API

### Option A: Using a config.ini file

Create a `config.ini` file containing your Anthropic API credentials:

```
[ANTHROPIC_API]
ANTHROPIC_KEY = your_api_key_here
```

### Option B: Using an environment variable (recommended)

```bash
export ANTHROPIC_API_KEY=your_api_key_here
```

If using the environment variable, the Anthropic SDK will pick it up automatically.

In [ ]:
# Option A: Loading API key from configuration file
# config = configparser.ConfigParser()
# config.read('../config.ini')
# ANTHROPIC_KEY = config.get('ANTHROPIC_API', 'ANTHROPIC_KEY')
# client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

# Option B: Using environment variable (recommended)
# The SDK automatically reads the ANTHROPIC_API_KEY environment variable
client = anthropic.Anthropic()

### Messages API : Get Claude's response to your prompt

#### Documentation : https://docs.anthropic.com/en/docs/build-with-claude/overview

In [ ]:
message_text = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"
company = "Amazon"

print(f"Tweet: {message_text} \nCompany: {company}")

In [ ]:
instruction = f"""\
You are a chatbot answering customer's messages. You are working for a company called {company}. Reply to the message below.
#####
Message:
\"{message_text}\" """
print(f"Instruction:\n\n{instruction}")

In [ ]:
response = client.messages.create(
    model="claude-haiku-3-5-20241022",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": instruction}
    ],
    temperature=0.7  # temperature ranges from 0 (deterministic) to 1 (creative)
)

In [ ]:
generated_text_vanilla = response.content[0].text
print(f"Answer generated:\n\n{generated_text_vanilla}")

### Embedding model : Get the embedding of a text

**Note**: Unlike OpenAI, Anthropic does not provide a built-in embedding API. For embeddings, we use open-source models. Here we use `sentence-transformers` which runs locally and is free.

In [ ]:
# Install sentence-transformers if not already installed
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer

# Load a lightweight embedding model (runs locally, no API cost)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
message_1 = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"
message_2 = "Ordered this around 2am Friday morning and it made it here already... good job @115830 https://t.co/XXMuII3QwQ"
message_3 = "I hate Amazon!!!"

In [ ]:
def get_embedding(text):
    """Get the embedding of an input text using a local model"""
    return embed_model.encode(text)

In [ ]:
embedding_message_1 = get_embedding(message_1)
embedding_message_2 = get_embedding(message_2)
embedding_message_3 = get_embedding(message_3)

In [ ]:
print(f"Message 1 : {message_1}\nEmbedding length : {len(embedding_message_1)}\nEmbedding (first 10 values):\n{embedding_message_1[:10]}")

#### Compare two embeddings to find similarity

A cosine similarity close to 1 implies that the sentence embeddings are very similar, meaning their vectors point in almost the same direction. This suggests the sentences have similar meanings or semantic content.

A cosine similarity around 0 indicates that the sentence embeddings are orthogonal (or near-orthogonal) to each other in the vector space, suggesting that the sentences are unrelated or have neutral similarity.

A cosine similarity close to -1 indicates that the embeddings are diametrically opposed in the vector space, suggesting that the sentences are highly dissimilar or have opposite meanings.

In [ ]:
def cosine_similarity(A, B):
    dot_product = np.dot(A, B)
    magnitude_A = np.linalg.norm(A)
    magnitude_B = np.linalg.norm(B)
    return dot_product / (magnitude_A * magnitude_B)

In [ ]:
similarity_message1_message2 = cosine_similarity(embedding_message_1, embedding_message_2)
print(f"Message 1 : {message_1}\nMessage 2 : {message_2}\n\nSimilarity: {similarity_message1_message2}")

In [ ]:
similarity_message1_message3 = cosine_similarity(embedding_message_1, embedding_message_3)
print(f"Message 1 : {message_1}\nMessage 3 : {message_3}\n\nSimilarity: {similarity_message1_message3}")

### Augment your prompt : RAG principle

In [ ]:
# Customer's Tweet to answer 
tweet = "Placed an order for some headphones early Monday morning, and they've arrived in 2 days... impressive service!"

In [ ]:
# Similar message with the agent's reply

similar_message = data_sample.head(1).customer_tweet[0]
reply = data_sample.head(1).company_tweet[0]
company = data_sample.head(1).company[0]

print(f"Customer's Tweet: {similar_message}\nCompany's Tweet: {reply}\nCompany: {company}")

In [ ]:
instruction = f"""\
You are a chatbot answering customer's tweet. You are working for a company called Amazon. 
You are provided with an example of a similar interaction between a customer and an agent. Reply to the customer's tweet in the same tone, structure and style than the provided example.

#####
Example :
Customer : \"{similar_message}\"
Agent : \"{reply}\"

#####
Tweet:
\"{tweet}\"
"""
print(f"Instruction:\n\n{instruction}")

In [ ]:
response = client.messages.create(
    model="claude-haiku-3-5-20241022",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": instruction}
    ],
    temperature=0.7
)

In [ ]:
generated_text = response.content[0].text
print(f"Answer generated with RAG:\n\n{generated_text}")

In [ ]:
# Vanilla answer without augmentation

print(f"Vanilla answer without augmentation:\n\n{generated_text_vanilla}")

### Bonus: Using Claude with Tool Use (Function Calling)

One of Claude's powerful features is **tool use** — the ability to define functions that Claude can call. This is the foundation for building AI Agents (Project 2).

In [ ]:
# Define a tool that Claude can use
tools = [
    {
        "name": "get_customer_info",
        "description": "Look up customer information by their username or ID. Returns customer name, account status, and recent orders.",
        "input_schema": {
            "type": "object",
            "properties": {
                "username": {
                    "type": "string",
                    "description": "The customer's username or Twitter handle"
                }
            },
            "required": ["username"]
        }
    }
]

response = client.messages.create(
    model="claude-haiku-3-5-20241022",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "Can you look up the customer info for @john_doe123? They seem to have an issue with their order."}
    ]
)

print("Response stop reason:", response.stop_reason)
for block in response.content:
    print(f"Block type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Input: {block.input}")
    elif block.type == "text":
        print(f"  Text: {block.text}")